<a href="https://colab.research.google.com/github/raki-rankawat/stm32-thesis/blob/main/Model_Comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =====================================================
# Imports
# =====================================================

import os
import time
import tarfile
import random
import shutil
from pathlib import Path
from urllib.request import urlretrieve
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from google.colab import drive

In [2]:
# =====================================================
# Mount Google Drive
# =====================================================

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# =====================================================
# Device Setup
# =====================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(41)
random.seed(41)

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

print("Device:", device)

Device: cuda


In [4]:
# =====================================================
# Auto Download + Prepare VWW (10k subset)
# =====================================================

vww_url = "https://www.silabs.com/public/files/github/machine_learning/benchmarks/datasets/vw_coco2014_96.tar.gz"

base_dir = Path("/content/vww_work")
archive_path = base_dir / "vw_coco2014_96.tar.gz"
extract_dir = base_dir / "extracted"
subset_dir = base_dir / "vww_10k"

# Subset config: 5k person + 5k non_person
n_per_class = 5000
val_ratio = 0.20


def download_vww():
    base_dir.mkdir(parents=True, exist_ok=True)

    if archive_path.exists() and archive_path.stat().st_size > 0:
        print("✅ VWW archive already downloaded")
        return

    print("⬇️ Downloading VWW archive...")
    urlretrieve(vww_url, archive_path)
    print("✅ Download complete:", archive_path)


def extract_vww():
    extract_dir.mkdir(parents=True, exist_ok=True)

    if any(extract_dir.iterdir()):
        print("✅ VWW already extracted")
        return

    print("📦 Extracting VWW archive...")
    with tarfile.open(archive_path, "r:gz") as tar:
        tar.extractall(extract_dir)
    print("✅ Extraction complete:", extract_dir)


def find_vww_root():
    # Find folder that contains BOTH person/ and non_person/
    for p in extract_dir.rglob("person"):
        if p.is_dir() and (p.parent / "non_person").is_dir():
            return p.parent

    raise RuntimeError("❌ Could not find 'person' and 'non_person' directories under extracted dataset")


def list_images(folder):
    exts = {".jpg", ".jpeg", ".png"}
    return [p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in exts]


def make_vww_subset(src_root):
    # Skip if subset already exists
    if (subset_dir / "train" / "person").is_dir() and (subset_dir / "val" / "non_person").is_dir():
        print("✅ VWW 10k subset already exists:", subset_dir)
        return

    for split in ["train", "val"]:
        for c in ["person", "non_person"]:
            (subset_dir / split / c).mkdir(parents=True, exist_ok=True)

    person_imgs = list_images(src_root / "person")
    nonperson_imgs = list_images(src_root / "non_person")

    if len(person_imgs) < n_per_class or len(nonperson_imgs) < n_per_class:
        raise ValueError(
            f"❌ Not enough images:\n"
            f"person: {len(person_imgs)} (need {n_per_class})\n"
            f"non_person: {len(nonperson_imgs)} (need {n_per_class})"
        )

    random.shuffle(person_imgs)
    random.shuffle(nonperson_imgs)

    person_sel = person_imgs[:n_per_class]
    nonperson_sel = nonperson_imgs[:n_per_class]

    def split_list(lst, val_ratio):
        n_val = int(len(lst) * val_ratio)
        return lst[n_val:], lst[:n_val]  # train, val

    p_train, p_val = split_list(person_sel, val_ratio)
    n_train, n_val = split_list(nonperson_sel, val_ratio)

    def copy_files(files, dst_dir):
        for f in files:
            dst = dst_dir / f.name

            # avoid rare collisions
            if dst.exists():
                dst = dst_dir / (f"{f.parent.name}_{f.name}")

            shutil.copy2(f, dst)

    print("🧩 Creating VWW 10k subset...")
    copy_files(p_train, subset_dir / "train" / "person")
    copy_files(p_val,   subset_dir / "val"   / "person")
    copy_files(n_train, subset_dir / "train" / "non_person")
    copy_files(n_val,   subset_dir / "val"   / "non_person")
    print("✅ VWW subset created at:", subset_dir)


download_vww()
extract_vww()
vww_root = find_vww_root()
print("✅ Found VWW root:", vww_root)
make_vww_subset(vww_root)

⬇️ Downloading VWW archive...
✅ Download complete: /content/vww_work/vw_coco2014_96.tar.gz
📦 Extracting VWW archive...


/tmp/ipykernel_403/3505717079.py:38: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(extract_dir)


✅ Extraction complete: /content/vww_work/extracted
✅ Found VWW root: /content/vww_work/extracted/vw_coco2014_96
🧩 Creating VWW 10k subset...
✅ VWW subset created at: /content/vww_work/vww_10k


In [5]:
# =====================================================
# Data Loaders
# =====================================================

batch_size = 64
img_size = 96

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomResizedCrop(img_size, scale=(0.6, 1.0)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

test_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

train_data = datasets.ImageFolder(root=str(subset_dir / "train"), transform=train_transform)
test_data  = datasets.ImageFolder(root=str(subset_dir / "val"),   transform=test_transform)

train_loader = DataLoader(
    train_data,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_data,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Class mapping:", train_data.class_to_idx)

Class mapping: {'non_person': 0, 'person': 1}


In [11]:
# =====================================================
# MobileNetV2 Components
# =====================================================

class InvertedResidual(nn.Module):

    def __init__(self, in_channels, out_channels, stride, expand_ratio):
        super().__init__()

        hidden_dim = in_channels * expand_ratio
        self.use_residual = stride == 1 and in_channels == out_channels

        self.block = nn.Sequential(

            nn.Conv2d(in_channels, hidden_dim, 1, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU6(inplace=True),

            nn.Conv2d(
                hidden_dim,
                hidden_dim,
                3,
                stride=stride,
                padding=1,
                groups=hidden_dim,
                bias=False
            ),

            nn.BatchNorm2d(hidden_dim),
            nn.ReLU6(inplace=True),

            nn.Conv2d(hidden_dim, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
        )

    def forward(self, x):

        if self.use_residual:
            return x + self.block(x)

        return self.block(x)

In [12]:
# =====================================================
# MobileNetV2 Model
# =====================================================

class VWW_MobileNetV2(nn.Module):

    def __init__(self):
        super().__init__()

        self.initial = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU6(inplace=True)
        )

        self.features = nn.Sequential(
            InvertedResidual(32, 16, 1, 1),

            InvertedResidual(16, 24, 2, 6),
            InvertedResidual(24, 24, 1, 6),

            InvertedResidual(24, 32, 2, 6),
            InvertedResidual(32, 32, 1, 6),

            InvertedResidual(32, 64, 2, 6),
            InvertedResidual(64, 64, 1, 6),
        )

        self.final_conv = nn.Sequential(
            nn.Conv2d(64, 512, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU6(inplace=True)
        )

        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(512, 2)
        self._initialize_weights()

    def forward(self, x):
        x = self.initial(x)
        x = self.features(x)
        x = self.final_conv(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

    def _initialize_weights(self):

        for m in self.modules():

            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out')

            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.zeros_(m.bias)

In [13]:
# =====================================================
# VGG-Style CNN
# =====================================================

class VWW_VGGStyle(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(

            # Block 1 : 96 -> 48
            nn.Conv2d(3, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2),

            # Block 2 : 48 -> 24
            nn.Conv2d(32, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2),

            # Block 3 : 24 -> 12
            nn.Conv2d(64, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.Conv2d(128, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2),

            # Block 4 : 12 -> 6
            nn.Conv2d(128, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.Conv2d(256, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 6 * 6, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),

            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),

            nn.Linear(128, 2)
        )

        self._initialize_weights()

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

    def _initialize_weights(self):

        for m in self.modules():

            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')

            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.zeros_(m.bias)

In [14]:
# =====================================================
# Loss and Model Initialization
# =====================================================

criterion = nn.CrossEntropyLoss()

mobilenet = VWW_MobileNetV2().to(device)
mobilenet.load_state_dict(
    torch.load(
        "/content/drive/My Drive/Colab Notebooks/vww_mobilenetv2_model.pth",
        map_location=device
    )
)

vgg_model = VWW_VGGStyle().to(device)
vgg_model.load_state_dict(
    torch.load(
        "/content/drive/My Drive/Colab Notebooks/vww_vggstyle_model.pth",
        map_location=device
    )
)

<All keys matched successfully>

In [19]:
# =====================================================
# Parameter Count
# =====================================================

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("MobileNet Params:", count_parameters(mobilenet))
print("VGG Params:", count_parameters(vgg_model))

MobileNet Params: 151874
VGG Params: 5958242


In [15]:
# =====================================================
# Inference Latency
# =====================================================

def measure_latency(model, input_size=(1, 3, 96, 96), warmup=20, runs=100):
    model.eval()

    dummy = torch.randn(input_size).to(device)

    with torch.no_grad():
        for _ in range(warmup):
            _ = model(dummy)

        if device.type == "cuda":
            torch.cuda.synchronize()

        start = time.time()

        for _ in range(runs):
            _ = model(dummy)

        if device.type == "cuda":
            torch.cuda.synchronize()

        end = time.time()

    return (end - start) / runs


mobilenet_latency = measure_latency(mobilenet)
vgg_latency = measure_latency(vgg_model)

print("MobileNet Latency (seconds):", mobilenet_latency)
print("VGG Latency (seconds):", vgg_latency)

MobileNet Latency (seconds): 0.001992361545562744
VGG Latency (seconds): 0.0010333967208862305


In [16]:
# =====================================================
# Model Size
# =====================================================

mobilenet_path = "mobilenet_vww.pth"
vgg_path = "vgg_vww.pth"

torch.save(mobilenet.state_dict(), mobilenet_path)
torch.save(vgg_model.state_dict(), vgg_path)

print("MobileNet Size (MB):", os.path.getsize(mobilenet_path) / 1e6)
print("VGG Size (MB):", os.path.getsize(vgg_path) / 1e6)

MobileNet Size (MB): 0.681171
VGG Size (MB): 23.858625


In [17]:
# =====================================================
# Accuracy Evaluation
# =====================================================

def evaluate(model, loader, device):
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for X, y in loader:
            X = X.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            outputs = model(X)
            preds = torch.argmax(outputs, dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total


mobilenet_acc = evaluate(mobilenet, test_loader, device)
vgg_acc = evaluate(vgg_model, test_loader, device)

print("MobileNet Accuracy:", mobilenet_acc * 100)
print("VGG Accuracy:", vgg_acc * 100)

MobileNet Accuracy: 79.4
VGG Accuracy: 81.75


In [20]:
# =====================================================
# Final Comparison Summary
# =====================================================

print("\n========== Final Comparison ==========")
print(f"MobileNet Accuracy   : {mobilenet_acc * 100:.2f}%")
print(f"VGG Accuracy         : {vgg_acc * 100:.2f}%")
print(f"MobileNet Params     : {count_parameters(mobilenet):,}")
print(f"VGG Params           : {count_parameters(vgg_model):,}")
print(f"MobileNet Latency    : {mobilenet_latency:.6f} sec")
print(f"VGG Latency          : {vgg_latency:.6f} sec")
print(f"MobileNet Size       : {os.path.getsize(mobilenet_path) / 1e6:.3f} MB")
print(f"VGG Size             : {os.path.getsize(vgg_path) / 1e6:.3f} MB")


========== Final Comparison ==========
MobileNet Accuracy   : 79.40%
VGG Accuracy         : 81.75%
MobileNet Params     : 151,874
VGG Params           : 5,958,242
MobileNet Latency    : 0.001992 sec
VGG Latency          : 0.001033 sec
MobileNet Size       : 0.681 MB
VGG Size             : 23.859 MB
